In [1]:
import numpy as np
import pandas as pd
import os
from working_data import  alt_label_df as label_df, normalize_by_window, clean_cols, add_timing

In [2]:
NORMALIZING_WINDOW_SIZE = 6*24
LABELING_WINDOW_SIZE = 18
POSITIVE_SLOPE = 0.3
NEGATIVE_SLOPE = 0.7
STARTING_HOUR = 0
ENDING_HOUR = 25
LABEL_CUR_CANDLE_MULTIPLIER = 0
LABEL_MEAN_MULTIPLIER = 8
BATCH_SIZE = 128
NUM_TOKENS = 64
OTHER_TOKENS = 64
LOOKBACK_WINDOW = NUM_TOKENS + 1
D_MODEL = 480
FF_DIM = D_MODEL* 2
NUM_HEADS = D_MODEL//24
DRAWDOWN = 0.5
LABEL_LOOKBACK = 512

In [3]:
def compute_real_targets_single_df(
        df, 
        opportunity_index,
        mean_multiplier=4, 
        drawdown=1, 
        lookback_window=64):
    """
    Convert normalized TP/SL values to real prices using a single DataFrame with preserved window min/max.
    
    Parameters:
    - df: Normalized DataFrame (with original OHLC columns and preserved 'window_min'/'window_max')
    - opportunity_index: Index of the opportunity in `df`
    - mean_multiplier: TP multiplier from labeling (default=4)
    - drawdown: SL multiplier from labeling (default=1)
    - lookback_window: Lookback window for mean candle size (default=64)
    
    Returns:
    - (target_signal_orig, stop_loss_orig): Real TP/SL prices
    """
    # Validate opportunity index
    if opportunity_index < 0 or opportunity_index >= len(df):
        raise IndexError("opportunity_index out of DataFrame bounds")
    
    # Verify required columns exist
    required_cols = ['close_normalized', 'window_min', 'window_max', 
                    'close_normalized_for_label', 'open_normalized_for_label']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Fetch normalized values
    close_norm = df.at[opportunity_index, 'close_normalized']
    window_min = df.at[opportunity_index, 'window_min']
    window_max = df.at[opportunity_index, 'window_max']
    current_close = df.at[opportunity_index, 'close']
    
    # Compute mean candle size in normalized space
    start_idx = max(0, opportunity_index - lookback_window)
    end_idx = opportunity_index
    if start_idx < end_idx:
        candles = (
            df['close_normalized_for_label'].iloc[start_idx:end_idx] - 
            df['open_normalized_for_label'].iloc[start_idx:end_idx]
        )
        mean_candle = candles.abs().mean()
    else:
        mean_candle = 0.0
    
    # Handle edge case: zero-width price window
    if window_max == window_min:
        return (float('nan'), float('nan'))
    
    # Calculate normalized TP/SL
    target_signal_norm = close_norm + (mean_candle * mean_multiplier)
    stop_loss_norm = close_norm - (mean_candle * drawdown)
    
    # Convert to real prices
    target_signal_orig = target_signal_norm * (window_max - window_min) + window_min
    stop_loss_orig = stop_loss_norm * (window_max - window_min) + window_min
    
    return (current_close, target_signal_orig, stop_loss_orig)

In [4]:
source_csv = "olderdata/GBPUSD/five_minutes.csv"
source_2_csv = "olderdata/GBPUSD/hours.csv"
working_path = "working"

In [5]:
df = pd.read_csv(source_csv)
df = clean_cols(df)
df = add_timing(df)
df = normalize_by_window(
    df, 
    window_size=NORMALIZING_WINDOW_SIZE, 
    low_col='low',
    high_col='high',
    normalizing_cols=[
        'open',
        'high',
        'low',
        'close'
    ],
    label_cols=['open', 'close'])
print("labeling")
df = label_df(df, window_size=LABELING_WINDOW_SIZE, 
              mean_multiplier=LABEL_MEAN_MULTIPLIER, 
              cur_candle_multiplier=LABEL_CUR_CANDLE_MULTIPLIER, 
              positive_slope=POSITIVE_SLOPE, 
              negative_slope=NEGATIVE_SLOPE,
              starting_hour=STARTING_HOUR,
              ending_hour=ENDING_HOUR,
              drawdown=DRAWDOWN,
              lookback_window=LABEL_LOOKBACK)

datetime
0     25083
5     25083
15    25082
25    25082
30    25082
35    25082
40    25082
45    25082
50    25082
10    25081
20    25081
55    25080
Name: count, dtype: int64
position_in_hour
0.000000    25083
0.090909    25083
0.272727    25082
0.454545    25082
0.545455    25082
0.636364    25082
0.727273    25082
0.818182    25082
0.909091    25082
0.181818    25081
0.363636    25081
1.000000    25080
Name: count, dtype: int64
labeling


In [6]:
df.reset_index()

,index,time,open,high,low,close,datetime,hour_of_day,position_in_hour,window_min,window_max,open_normalized,high_normalized,low_normalized,close_normalized,open_normalized_for_label,close_normalized_for_label,target,include
0,0,1609761600,1.36719,1.36804,1.36698,1.36789,2021-01-04 12:00:00,12,0.000000,1.36417,1.37022,0.499174,0.639669,0.464463,0.614876,0.499174,0.614876,0,1
1,1,1609761900,1.36789,1.36807,1.36754,1.36798,2021-01-04 12:05:00,12,0.090909,1.36417,1.37022,0.614876,0.644628,0.557025,0.629752,0.614876,0.629752,0,1
2,2,1609762200,1.36798,1.36824,1.36769,1.36816,2021-01-04 12:10:00,12,0.181818,1.36455,1.37022,0.604938,0.650794,0.553792,0.636684,0.629752,0.659504,0,1
3,3,1609762500,1.36816,1.36837,1.36740,1.36763,2021-01-04 12:15:00,12,0.272727,1.36460,1.37022,0.633452,0.670819,0.498221,0.539146,0.636684,0.543210,0,1
4,4,1609762800,1.36758,1.36818,1.36758,1.36817,2021-01-04 12:20:00,12,0.363636,1.36475,1.37022,0.517367,0.627057,0.517367,0.625229,0.530249,0.635231,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300815,300815,1736882400,1.21804,1.21810,1.21750,1.21810,2025-01-14 19:20:00,19,0.363636,1.21365,1.22448,0.405355,0.410896,0.355494,0.410896,0.405355,0.410896,0,1
300816,300816,1736882700,1.21810,1.21829,1.21789,1.21797,2025-01-14 19:25:00,19,0.454545,1.21365,1.22448,0.410896,0.428440,0.391505,0.398892,0.410896,0.398892,0,1
300817,300817,1736883000,1.21799,1.21837,1.21791,1.21807,2025-01-14 19:30:00,19,0.545455,1.21365,1.22448,0.400739,0.435826,0.393352,0.408126,0.400739,0.408126,0,1
300818,300818,1736883300,1.21807,1.21825,1.21774,1.21821,2025-01-14 19:35:00,19,0.636364,1.21365,1.22448,0.408126,0.424746,0.377655,0.421053,0.408126,0.421053,0,1


In [7]:
df[df['target'] == 1].index

Index([   193,    195,    196,    197,    198,    200,    201,    202,    203,
          288,
       ...
       299723, 299731, 299732, 299896, 300044, 300167, 300168, 300494, 300505,
       300703],
      dtype='int64', length=3888)

In [8]:
compute_real_targets_single_df(df, opportunity_index=193, mean_multiplier=LABEL_MEAN_MULTIPLIER, drawdown=DRAWDOWN, lookback_window=LABEL_LOOKBACK)

(1.35687, 1.3597844009305258, 1.3566878499418422)

In [9]:
df[193:193+LABELING_WINDOW_SIZE+1][['open', 'high', 'low', 'close', 'close_normalized', 'window_min', 'window_max']]

,open,high,low,close,close_normalized,window_min,window_max
193,1.35696,1.35700,1.35666,1.35687,0.314444,1.35404,1.36304
194,1.35687,1.35742,1.35687,1.35732,0.364444,1.35404,1.36304
195,1.35732,1.35746,1.35709,1.35736,0.368889,1.35404,1.36304
196,1.35736,1.35755,1.35728,1.35744,0.377778,1.35404,1.36304
197,1.35744,1.35786,1.35733,1.35779,0.416667,1.35404,1.36304
198,1.35781,1.35812,1.35776,1.35781,0.418889,1.35404,1.36304
199,1.35781,1.35841,1.35778,1.35841,0.485556,1.35404,1.36304
200,1.35841,1.35859,1.35820,1.35855,0.501111,1.35404,1.36304
201,1.35855,1.35866,1.35842,1.35858,0.504444,1.35404,1.36304
202,1.35858,1.35872,1.35851,1.35855,0.501111,1.35404,1.36304
